# Lab 1 — Your First AI API Call
**Generative AI: Foundations and Applications · Session 1 · TCE Madurai**

Before you start:
1. **File → Save a copy in Drive** (do it now, or your work vanishes).
2. Have your Gemini API key ready — from https://aistudio.google.com → *Get API key*.
3. Run cells top to bottom with **Shift+Enter**.

> Your key is a secret. This notebook asks for it with a hidden input box — never paste it into a code cell.

In [ ]:
# Cell 1 — install the SDK (takes ~20 seconds)
%pip install -q -U google-genai
print("SDK installed ✓")

In [ ]:
# Cell 2 — enter your API key (input stays hidden)
from getpass import getpass
from google import genai

API_KEY = getpass("Paste your Gemini API key and press Enter: ")
client = genai.Client(api_key=API_KEY)

# One variable controls which model we use everywhere.
# Free tier friendly. If a newer model is free when you read this, change this one line.
# Current list: https://ai.google.dev/gemini-api/docs/models
MODEL = "gemini-flash-latest"  # the free tier's current Flash (July 2026 → Gemini 3.5 Flash). 503 'high demand'? swap to "gemini-flash-lite-latest".
print("Client ready ✓  using model:", MODEL)

## Part B — Your first call

Four lines. That's all it takes to talk to a frontier model.

In [ ]:
# Cell 3 — first API call
response = client.models.generate_content(
    model=MODEL,
    contents="Introduce yourself in 2 sentences to a class of engineering students in Madurai."
)
print(response.text)

### ✓ Checkpoint 1
If you see a response above — congratulations, you are officially calling one of the most capable AI models on Earth from your own code. Read it aloud to your partner.

---
## A helper function (with rate-limit protection)

The free tier allows ~10 requests/minute. If the whole class hits the API at once you may see a `429` error — the helper below waits and retries automatically.

In [ ]:
# Cell 4 — helper with retry
import time

def ask(prompt, temperature=None, model=None):
    """Send a prompt to Gemini, retrying politely if we hit rate limits."""
    from google.genai import types
    config = types.GenerateContentConfig(temperature=temperature) if temperature is not None else None
    for attempt in range(4):
        try:
            r = client.models.generate_content(
                model=model or MODEL, contents=prompt, config=config)
            return r.text
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                wait = 20 * (attempt + 1)
                print(f"Rate limited — waiting {wait}s (attempt {attempt+1}/3)...")
                time.sleep(wait)
            else:
                raise
print("Helper ready ✓")

## Part C — The five prompts

Five core skills: explain, summarize, translate, extract, roleplay. Run the cell, read every output carefully.

In [ ]:
# Cell 5 — five prompts
prompts = {
    "1 · EXPLAIN":   "Explain how UPI works to a 10-year-old, in 5 sentences.",
    "2 · SUMMARIZE": "Summarize the plot of Ponniyin Selvan in exactly 3 bullet points.",
    "3 · TRANSLATE": "Translate to formal Tamil: 'The exam has been postponed to next Monday.'",
    "4 · EXTRACT":   "Extract name, degree, year as JSON from: 'Hi, I'm Priya, third year BE CSE at TCE.'",
    "5 · ROLEPLAY":  "You are a strict interviewer at a product company. Ask me one DSA question, then wait for my answer.",
}

for label, p in prompts.items():
    print("=" * 70)
    print(label, "→", p)
    print("-" * 70)
    print(ask(p))
    print()

### Your observations (edit this cell — double-click)

One honest line per prompt: what was good, what was off?

| # | Good | Off / surprising |
|---|------|------------------|
| 1 Explain | | |
| 2 Summarize | | |
| 3 Translate | *(ask a Tamil speaker: formal enough?)* | |
| 4 Extract | *(valid JSON? extra text around it?)* | |
| 5 Roleplay | | |

### ✓ Checkpoint 2 — all five ran, five observations written.

---
## Part D — One prompt, three models

Pick ONE prompt (from above, or your own). Run it here on Gemini, then paste the same text into:
- **ChatGPT** → https://chatgpt.com
- **One more**: Claude (https://claude.ai) / Copilot / Meta AI

In [ ]:
# Cell 6 — your chosen prompt on Gemini
my_prompt = "Explain how UPI works to a 10-year-old, in 5 sentences."   # ← change me

print(ask(my_prompt))

### Comparison table (edit this cell)

| | Gemini (API) | ChatGPT | Third model: ______ |
|---|---|---|---|
| Length / format | | | |
| Tone / personality | | | |
| Accuracy issues? | | | |
| Ship it to a user? | | | |

**The one difference that surprised me most:** _______________

### ✓ Checkpoint 3 — show the instructor your Gemini output + your surprise.

---
## Stretch goals

In [ ]:
# Stretch 1 — Temperature: robotic vs poetic
p = "Give a creative name for a new juice shop near TCE, one name only."

print("--- temperature = 0.0 (three runs) ---")
for i in range(3):
    print(f"  run {i+1}:", ask(p, temperature=0.0))

print("--- temperature = 1.5 (three runs) ---")
for i in range(3):
    print(f"  run {i+1}:", ask(p, temperature=1.5))

# What do you notice? Which setting for a legal document? Which for a movie script?

In [ ]:
# Stretch 2 — Tamil stress test
q_en = "Who composed the music for the film 'Roja' and in which year was it released?"
q_ta = "'ரோஜா' திரைப்படத்திற்கு இசையமைத்தவர் யார்? அது எந்த ஆண்டு வெளியானது?"

print("EN →", ask(q_en))
print()
print("TA →", ask(q_ta))

# Same facts? Same quality? Same length?

In [ ]:
# Stretch 3 — Count tokens (remember the lecture demo?)
en = "The exam has been postponed to next Monday."
ta = "தேர்வு அடுத்த திங்கட்கிழமைக்கு ஒத்திவைக்கப்பட்டுள்ளது."

for label, text in [("English", en), ("Tamil  ", ta)]:
    n = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f"{label}: {n:3d} tokens ← {text}")

# Same meaning. Compare the counts — this is the tokenizer equity issue, measured by you.

### S4 · The 15-line language model (the lecture's counting demo)

Remember *“those odds aren’t magic — you just count words”*? Here is that whole idea as runnable code. You **train** a next-word model by tallying, then **sample** sentences from it — the same three steps Gemini uses (read context → get a distribution → sample), except here step 2 is a table you can print, and “training” is pure counting.

In [ ]:
# Stretch 4 — the whole idea in ~15 lines: a next-word model you can read.
# Same three steps as Gemini (context -> distribution -> sample); "trained" by counting.
import random
from collections import defaultdict, Counter

corpus = [
    "the build is failing again",
    "the build is passing now",
    "did you push the code",
    "did you fix the bug",
    "i think the code is fine",
]

table = defaultdict(Counter)                 # "training" = tally which word follows which
for msg in corpus:
    words = ["<start>"] + msg.split() + ["<end>"]
    for a, b in zip(words, words[1:]):
        table[a][b] += 1

def next_word(word):                         # read one row, roll weighted by the counts
    choices = table[word]
    return random.choices(list(choices), weights=list(choices.values()))[0]

def generate():
    word, out = "<start>", []
    while (word := next_word(word)) != "<end>":
        out.append(word)
    return " ".join(out)

for _ in range(5):
    print(generate())

print("\nThat's a language model. Gemini runs the same loop — but step 2 is a")
print("learned function over billions of parameters, not a table you can print.")

## Part E — Build your test set (last 10 min)

Sessions run back-to-back, so do this **now**:

1. Pick a subject you know **cold** (DSA, cricket, cinema, your hometown — anything).
2. Write **10 questions + correct answers** in a text file and keep it handy.
3. That file becomes your test set in the very next session — the lie-detector lab.

## Overnight (before Day 2 — 5 min)

Day 2 = **chat with your own notes** (RAG). Put 2–3 real documents on your laptop or Drive: lecture notes, a textbook chapter PDF, anything you'd genuinely want to query.

## Remember
- Never share or commit your API key.
- Free-tier data may be used by Google to improve products — don't paste anything private.
- Rate limits: ~10 requests/min. The `ask()` helper handles the occasional 429.

**You just did real AI engineering. Short break — then Session 2: catching AI lying, with numbers.**